# Virtual Diabetes Advisor — Risk Model (Team 6)

**Course:** Machine Learning mini project · **Dataset:** CDC BRFSS 2015 Diabetes Health Indicators

The advisor computes a *personal* glucose threshold for every user and warns them when a reading
enters a 5 % pre-breach band. This notebook builds the ML part of that system: a model that turns a
user's health profile into a **calibrated diabetes-risk probability and risk tier**. The tier then
tightens or relaxes the user's personal threshold (requirement R2).

| Section | Syllabus topic |
|---|---|
| 1–3 | Data loading, validation, exploratory analysis |
| 4 | **Maximum Likelihood Estimation** — Gaussian, Bernoulli, logistic regression from scratch |
| 5 | **Linear regression** — normal equation vs scikit-learn, diagnostics |
| 6–8 | **Classification** — model comparison, honest evaluation, calibration, 3-class model |
| 9–10 | Explainability, conclusions and how the model plugs into the app |

> **Important limitation:** BRFSS is a telephone *survey*. It contains no glucose readings, so this
> model estimates **risk from lifestyle and health answers**; it does not read or predict blood sugar.
> Glucose readings in the app come from users and are checked against the threshold engine.

## 1. Setup

Run this notebook from the project's `notebooks/` folder (data is read from `../data/raw`).
On **Google Colab**, upload the three CSVs to the session first; the path is detected automatically.

In [ ]:
# Standard library
from pathlib import Path

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd
from scipy import optimize, stats
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    brier_score_loss,
    classification_report,
    f1_score,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FloatArray = npt.NDArray[np.float64]
RANDOM_STATE = 42  # one seed for every split and model -> reproducible results
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.precision", 4)

# Repo layout: notebooks/ sits next to data/raw. On Colab, CSVs are uploaded to the cwd.
DATA_DIR = next(
    (p for p in (Path("../data/raw"), Path("data/raw"), Path(".")) if
     (p / "diabetes_binary_health_indicators_BRFSS2015.csv").exists()),
    Path("."),
)
print("Reading data from:", DATA_DIR.resolve())

## 2. Load and validate the three files

| File | Label | Purpose here |
|---|---|---|
| `diabetes_binary_5050split…` | 0/1, balanced | fast model comparison (Section 6) |
| `diabetes_binary…` | 0/1, real prevalence | honest evaluation + calibration (Section 7) |
| `diabetes_012…` | 0 none / 1 prediabetes / 2 diabetes | multi-class model (Section 8) |

In [ ]:
FILES = {
    "binary": "diabetes_binary_health_indicators_BRFSS2015.csv",
    "binary_5050": "diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
    "three_class": "diabetes_012_health_indicators_BRFSS2015.csv",
}
raw = {name: pd.read_csv(DATA_DIR / fname) for name, fname in FILES.items()}

summary = pd.DataFrame(
    {
        name: {
            "rows": len(df),
            "columns": df.shape[1],
            "null values": int(df.isna().sum().sum()),
            "duplicate rows": int(df.duplicated().sum()),
            "label": df.columns[0],
        }
        for name, df in raw.items()
    }
).T
summary

In [ ]:
FEATURES = [c for c in raw["binary"].columns if c != "Diabetes_binary"]
CONTINUOUS = ["BMI", "MentHlth", "PhysHlth"]
ORDINAL = ["GenHlth", "Age", "Education", "Income"]
BINARY_COLS = [c for c in FEATURES if c not in CONTINUOUS + ORDINAL]

# Valid ranges from the BRFSS codebook; anything outside means corrupted data.
RANGES = {**{c: (0, 1) for c in BINARY_COLS}, "BMI": (12, 98), "MentHlth": (0, 30),
          "PhysHlth": (0, 30), "GenHlth": (1, 5), "Age": (1, 13), "Education": (1, 6),
          "Income": (1, 8)}
for name, df in raw.items():
    bad = {c: int(((df[c] < lo) | (df[c] > hi)).sum()) for c, (lo, hi) in RANGES.items()}
    assert sum(bad.values()) == 0, f"{name}: out-of-range values {bad}"
print(f"{len(FEATURES)} features, all values inside codebook ranges, no nulls.")

# Identical survey answers would land in both train and test and inflate scores -> drop them.
data = {name: df.drop_duplicates().reset_index(drop=True).astype(float) for name, df in raw.items()}
{name: len(df) for name, df in data.items()}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (name, df) in zip(axes, data.items()):
    label = df.columns[0]
    share = df[label].value_counts(normalize=True).sort_index() * 100
    ax.bar(share.index.astype(int).astype(str), share.values, color="#4C72B0")
    ax.set_title(f"{name}  ({label})")
    ax.set_ylabel("% of rows")
    for x, v in zip(share.index.astype(int).astype(str), share.values):
        ax.text(x, v + 1, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()

**Observation:** in the real-prevalence file only ~14 % of people are diabetic (~15 % after removing duplicates), and in the 3-class
file prediabetes is under 2 %. Accuracy is therefore a misleading metric (predicting "no diabetes"
for everyone already scores 86 %). We evaluate with **recall, precision, F1, ROC-AUC and PR-AUC**.

## 3. Exploratory data analysis

In [ ]:
df = data["binary"]
y = df["Diabetes_binary"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# BMI distribution by class
for label, color in [(0, "#4C72B0"), (1, "#DD8452")]:
    axes[0].hist(df.loc[y == label, "BMI"].clip(upper=60), bins=40, alpha=0.6, density=True,
                 color=color, label=f"Diabetes = {label}")
axes[0].set(title="BMI by diabetes status", xlabel="BMI (clipped at 60)", ylabel="density")
axes[0].legend()

# Diabetes rate per age bucket (1 = 18-24 ... 13 = 80+)
rate_age = df.groupby("Age")["Diabetes_binary"].mean() * 100
axes[1].bar(rate_age.index.astype(int), rate_age.values, color="#4C72B0")
axes[1].set(title="Diabetes rate by age bucket", xlabel="Age bucket", ylabel="% diabetic")

# Diabetes rate by self-rated general health (1 excellent ... 5 poor)
rate_gen = df.groupby("GenHlth")["Diabetes_binary"].mean() * 100
axes[2].bar(rate_gen.index.astype(int), rate_gen.values, color="#4C72B0")
axes[2].set(title="Diabetes rate by general health", xlabel="1 = excellent … 5 = poor",
            ylabel="% diabetic")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each feature with the label (point-biserial for 0/1 vs numeric).
corr = df[FEATURES].corrwith(y).sort_values()
colors = ["#DD8452" if v > 0 else "#4C72B0" for v in corr.values]
plt.figure(figsize=(8, 6))
plt.barh(corr.index, corr.values, color=colors)
plt.title("Correlation with Diabetes_binary")
plt.xlabel("Pearson r")
plt.tight_layout()
plt.show()
corr.sort_values(ascending=False).head(6)

**Observations**
* General health, high blood pressure, BMI, difficulty walking, high cholesterol and age have the
  strongest positive association with diabetes.
* Income, education and physical activity are negatively associated (protective / socioeconomic).
* No single feature separates the classes — a multivariate model is needed.

## 4. Maximum Likelihood Estimation (MLE)

Given data $x_1,\dots,x_n$ and a model $p(x\mid\theta)$, MLE picks
$$\hat\theta = \arg\max_\theta \; \ell(\theta) = \sum_{i=1}^n \log p(x_i \mid \theta).$$

We show MLE three ways: a closed-form Gaussian, a Bernoulli, and logistic regression
(no closed form → numerical optimisation).

### 4.1 Gaussian MLE for BMI

For $x \sim \mathcal N(\mu, \sigma^2)$:
$\ell(\mu,\sigma) = -\tfrac n2\log(2\pi\sigma^2) - \tfrac1{2\sigma^2}\sum (x_i-\mu)^2$.
Setting the derivatives to zero gives $\hat\mu = \bar x$ and $\hat\sigma^2 = \tfrac1n\sum (x_i-\bar x)^2$
(note $1/n$, not $1/(n-1)$ — the MLE variance is biased). We verify this numerically.

In [ ]:
bmi: FloatArray = df["BMI"].to_numpy()


def gaussian_neg_log_likelihood(params: FloatArray, x: FloatArray) -> float:
    '''Negative Gaussian log-likelihood.

    Args:
        params: ``[mu, log_sigma]``; optimising log(sigma) keeps sigma positive.
        x: Observations.

    Returns:
        ``-sum(log N(x | mu, sigma^2))``.
    '''
    mu, log_sigma = params
    return float(-stats.norm.logpdf(x, loc=mu, scale=np.exp(log_sigma)).sum())


result = optimize.minimize(gaussian_neg_log_likelihood, x0=np.array([20.0, 1.0]), args=(bmi,))
mu_numeric, sigma_numeric = result.x[0], float(np.exp(result.x[1]))

pd.DataFrame(
    {
        "closed form": [bmi.mean(), bmi.std(ddof=0)],
        "numerical optimisation": [mu_numeric, sigma_numeric],
        "unbiased sample std (ddof=1)": [np.nan, bmi.std(ddof=1)],
    },
    index=["mu", "sigma"],
)

In [ ]:
# BMI is right-skewed, so compare the Gaussian with a log-normal fitted by MLE.
shape, loc, scale = stats.lognorm.fit(bmi, floc=0)  # scipy's fit() is an MLE
ll_normal = stats.norm.logpdf(bmi, bmi.mean(), bmi.std(ddof=0)).sum()
ll_lognormal = stats.lognorm.logpdf(bmi, shape, loc, scale).sum()
# AIC = 2k - 2 log L, both models have k = 2 parameters; lower is better.
print(f"Gaussian   log-likelihood {ll_normal:,.0f}   AIC {4 - 2 * ll_normal:,.0f}")
print(f"Log-normal log-likelihood {ll_lognormal:,.0f}   AIC {4 - 2 * ll_lognormal:,.0f}")

grid = np.linspace(12, 70, 400)
plt.hist(bmi, bins=80, range=(12, 70), density=True, alpha=0.5, label="BMI data")
plt.plot(grid, stats.norm.pdf(grid, bmi.mean(), bmi.std(ddof=0)), label="Gaussian MLE")
plt.plot(grid, stats.lognorm.pdf(grid, shape, loc, scale), label="Log-normal MLE")
plt.title("MLE fits to BMI")
plt.xlabel("BMI")
plt.legend()
plt.show()

The log-normal has a much higher likelihood (lower AIC): BMI is skewed, which is also why the
pipeline uses a **RobustScaler** for BMI in linear models.

### 4.2 Bernoulli MLE for diabetes prevalence

For labels $y_i\in\{0,1\}$ with $P(y=1)=p$: $\ell(p)=\sum y_i\log p+(1-y_i)\log(1-p)$, maximised at
$\hat p=\bar y$.

In [ ]:
labels: FloatArray = y.to_numpy()
p_grid = np.linspace(0.01, 0.99, 500)
log_lik = labels.sum() * np.log(p_grid) + (len(labels) - labels.sum()) * np.log(1 - p_grid)
p_hat_grid = p_grid[np.argmax(log_lik)]

plt.plot(p_grid, log_lik)
plt.axvline(labels.mean(), color="red", linestyle="--", label=f"sample mean = {labels.mean():.4f}")
plt.title("Bernoulli log-likelihood of diabetes prevalence")
plt.xlabel("p")
plt.ylabel("log-likelihood")
plt.legend()
plt.show()
print(f"argmax on grid: {p_hat_grid:.4f}   closed-form MLE (mean): {labels.mean():.4f}")

### 4.3 Logistic regression *is* MLE

Logistic regression models $P(y=1\mid x)=\sigma(w^\top x+b)$ and fits $w,b$ by maximising the
Bernoulli log-likelihood
$\ell(w)=\sum y_i\log\sigma(z_i)+(1-y_i)\log(1-\sigma(z_i))$. There is no closed form, but the
gradient is simple: $\nabla_w \ell = X^\top(y-\sigma(Xw))$. We implement gradient ascent from scratch
on the balanced 50/50 file and check it against scikit-learn's (unregularised) solver.

In [ ]:
def sigmoid(z: FloatArray) -> FloatArray:
    '''Numerically stable logistic function.'''
    return 0.5 * (1.0 + np.tanh(0.5 * z))


def fit_logistic_mle(
    x: FloatArray, y_true: FloatArray, lr: float = 0.5, epochs: int = 3000
) -> tuple[FloatArray, list[float]]:
    '''Fit logistic regression by gradient ascent on the mean log-likelihood.

    Args:
        x: Standardised features, shape (n, d). A bias column is added internally.
        y_true: 0/1 labels, shape (n,).
        lr: Learning rate.
        epochs: Number of full-batch gradient steps.

    Returns:
        Weights of shape (d + 1,) with the bias first, and the log-likelihood history.
    '''
    xb = np.column_stack([np.ones(len(x)), x])
    w = np.zeros(xb.shape[1])
    history: list[float] = []
    for _ in range(epochs):
        p = np.clip(sigmoid(xb @ w), 1e-12, 1 - 1e-12)
        history.append(float(np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))))
        w += lr * xb.T @ (y_true - p) / len(y_true)  # gradient of the mean log-likelihood
    return w, history


balanced = data["binary_5050"]
x_bal = StandardScaler().fit_transform(balanced[FEATURES])
y_bal = balanced["Diabetes_binary"].to_numpy()

w_scratch, ll_history = fit_logistic_mle(x_bal, y_bal)
sk = LogisticRegression(penalty=None, max_iter=5000).fit(x_bal, y_bal)

comparison = pd.DataFrame(
    {"from scratch (MLE)": w_scratch[1:], "scikit-learn": sk.coef_[0]}, index=FEATURES
)
comparison.loc["(intercept)"] = [w_scratch[0], sk.intercept_[0]]
print("Max absolute difference:", float(np.abs(comparison.diff(axis=1).iloc[:, 1]).max()))
comparison.sort_values("scikit-learn", key=np.abs, ascending=False).head(10)

In [ ]:
plt.plot(ll_history)
plt.title("Gradient ascent: mean log-likelihood per epoch")
plt.xlabel("epoch")
plt.ylabel("mean log-likelihood")
plt.show()

Both solvers reach the same coefficients — scikit-learn's `LogisticRegression` is a maximum-likelihood
estimator. Positive weights (e.g. GenHlth, BMI, HighBP, Age) raise the log-odds of diabetes.

## 5. Linear regression — predicting BMI

Task: predict a person's BMI from the other answers. Under the assumption
$y = Xw + \varepsilon,\ \varepsilon\sim\mathcal N(0,\sigma^2)$, maximising the likelihood is the same
as minimising squared error, giving the **normal equation** $\hat w = (X^\top X)^{-1}X^\top y$
(OLS = MLE under Gaussian noise — linking back to Section 4).

In [ ]:
reg_features = [c for c in FEATURES if c != "BMI"] + ["Diabetes_binary"]
x_reg = df[reg_features].to_numpy()
y_reg = df["BMI"].to_numpy()
xr_train, xr_test, yr_train, yr_test = train_test_split(
    x_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

# Normal equation; lstsq solves it stably without forming an explicit inverse.
xb_train = np.column_stack([np.ones(len(xr_train)), xr_train])
w_ols, *_ = np.linalg.lstsq(xb_train, yr_train, rcond=None)

lin = LinearRegression().fit(xr_train, yr_train)
pred = lin.predict(xr_test)
residuals = yr_test - pred

print(f"Max |normal equation - sklearn| coefficient difference: "
      f"{np.abs(w_ols[1:] - lin.coef_).max():.2e}")
print(f"Test R^2  : {r2_score(yr_test, pred):.3f}")
print(f"Test RMSE : {np.sqrt(mean_squared_error(yr_test, pred)):.2f} BMI points")
print(f"Baseline RMSE (predict the mean): {yr_test.std():.2f}")

pd.Series(lin.coef_, index=reg_features).sort_values(key=np.abs, ascending=False).head(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sample = np.random.default_rng(RANDOM_STATE).choice(len(pred), 4000, replace=False)
axes[0].scatter(pred[sample], residuals[sample], s=4, alpha=0.3)
axes[0].axhline(0, color="red")
axes[0].set(title="Residuals vs fitted", xlabel="predicted BMI", ylabel="residual")
axes[1].hist(residuals, bins=80, range=(-20, 40), density=True)
axes[1].set(title="Residual distribution", xlabel="residual")
plt.tight_layout()
plt.show()

**Interpretation:** R² is low — survey answers explain only a small part of BMI variation, and the
residuals are right-skewed (a few very high BMIs), so the Gaussian-noise assumption only partly holds.
The coefficients are still interpretable: having diabetes, high blood pressure, poor general health
and difficulty walking are associated with higher BMI, while physical activity is associated with lower BMI.
Linear regression is a diagnostic tool here, not part of the deployed system.

## 6. Classification — model comparison on the balanced file

Balanced data makes comparison fast and fair. Linear models get scaled inputs; trees do not need it.
Gradient boosting uses XGBoost if installed, otherwise scikit-learn's equivalent.

In [ ]:
def scores(y_true: FloatArray, proba: FloatArray, threshold: float = 0.5) -> dict[str, float]:
    '''Key binary-classification metrics at a probability threshold.

    Args:
        y_true: 0/1 labels.
        proba: Predicted probability of class 1.
        threshold: Cut-off for a positive prediction.

    Returns:
        recall, precision, F1, ROC-AUC, PR-AUC and Brier score.
    '''
    pred = (proba >= threshold).astype(int)
    return {
        "recall": recall_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
        "Brier": brier_score_loss(y_true, proba),
    }


def gradient_boosting(pos_weight: float = 1.0) -> object:
    '''XGBoost if available, else HistGradientBoostingClassifier (same idea, no extra install).'''
    try:
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5, subsample=0.8,
                             colsample_bytree=0.8, scale_pos_weight=pos_weight,
                             eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE)
    except ImportError:
        return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400,
                                              class_weight="balanced", random_state=RANDOM_STATE)


linear_preprocess = ColumnTransformer(
    [("continuous", RobustScaler(), CONTINUOUS), ("ordinal", StandardScaler(), ORDINAL)],
    remainder="passthrough",
)
models = {
    "Logistic regression": Pipeline([("prep", linear_preprocess),
                                     ("clf", LogisticRegression(max_iter=2000))]),
    "Decision tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=50,
                                            random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1,
                                            random_state=RANDOM_STATE),
    "Gradient boosting": gradient_boosting(),
}

xb_tr, xb_te, yb_tr, yb_te = train_test_split(
    balanced[FEATURES], balanced["Diabetes_binary"].astype(int), test_size=0.2,
    stratify=balanced["Diabetes_binary"], random_state=RANDOM_STATE,
)
balanced_probas: dict[str, FloatArray] = {}
rows = {}
for name, model in models.items():
    model.fit(xb_tr, yb_tr)
    balanced_probas[name] = model.predict_proba(xb_te)[:, 1]
    rows[name] = scores(yb_te.to_numpy(), balanced_probas[name])
pd.DataFrame(rows).T.sort_values("PR-AUC", ascending=False)

In [ ]:
for name, proba in balanced_probas.items():
    fpr, tpr, _ = roc_curve(yb_te, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(yb_te, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.title("ROC curves — balanced test set")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate (recall)")
plt.legend()
plt.show()

All four models land within about 0.015 ROC-AUC of each other (≈ 0.81–0.82). Gradient boosting is
best, but plain logistic regression is close behind, which tells us most of the signal is roughly
additive in log-odds. The single decision tree is the weakest. On balanced data precision looks
good (~0.73) — Section 7 shows why that number does not survive real prevalence.

## 7. Honest evaluation at real prevalence, calibration and threshold choice

The balanced file hides a real-world problem: with ~15 % positives, **precision falls sharply**.
So the deployed model is trained and evaluated on the real-prevalence file:

1. Train / validation / test = 70 / 15 / 15, stratified.
2. Train gradient boosting with the rare class up-weighted.
3. **Calibrate** with isotonic regression so a predicted 0.30 really means ~30 % of such people are
   diabetic — the threshold engine uses these probabilities directly.
4. Choose the decision threshold on validation by maximising **F2** (recall counts twice: missing a
   diabetic person is worse than a false alarm).

In [ ]:
full = data["binary"]
xf, yf = full[FEATURES], full["Diabetes_binary"].astype(int)
xf_rest, xf_test, yf_rest, yf_test = train_test_split(
    xf, yf, test_size=0.15, stratify=yf, random_state=RANDOM_STATE)
xf_train, xf_val, yf_train, yf_val = train_test_split(
    xf_rest, yf_rest, test_size=0.15 / 0.85, stratify=yf_rest, random_state=RANDOM_STATE)

prevalence = yf_train.mean()
pos_weight = (1 - prevalence) / prevalence
print(f"train {len(xf_train):,}  val {len(xf_val):,}  test {len(xf_test):,}  "
      f"prevalence {prevalence:.3f}")

uncalibrated = gradient_boosting(pos_weight).fit(xf_train, yf_train)
calibrated = CalibratedClassifierCV(gradient_boosting(pos_weight), method="isotonic", cv=3)
calibrated.fit(xf_train, yf_train)

proba_uncal = uncalibrated.predict_proba(xf_test)[:, 1]
proba_cal = calibrated.predict_proba(xf_test)[:, 1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, proba in [("Uncalibrated (class-weighted)", proba_uncal), ("Isotonic-calibrated", proba_cal)]:
    frac_pos, mean_pred = calibration_curve(yf_test, proba, n_bins=15, strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", label=name)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfect")
axes[0].set(title="Calibration (reliability) curve", xlabel="predicted probability",
            ylabel="observed diabetic share")
axes[0].legend()

precision, recall, _ = precision_recall_curve(yf_test, proba_cal)
axes[1].plot(recall, precision)
axes[1].axhline(yf_test.mean(), color="grey", linestyle="--", label=f"base rate {yf_test.mean():.2f}")
axes[1].set(title=f"Precision–recall (PR-AUC {average_precision_score(yf_test, proba_cal):.3f})",
            xlabel="recall", ylabel="precision")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"Brier score  uncalibrated {brier_score_loss(yf_test, proba_uncal):.4f}   "
      f"calibrated {brier_score_loss(yf_test, proba_cal):.4f}")

In [ ]:
def best_f_beta_threshold(y_true: FloatArray, proba: FloatArray, beta: float = 2.0) -> float:
    '''Threshold that maximises F-beta on the given (validation) data.'''
    p, r, t = precision_recall_curve(y_true, proba)
    p, r = p[:-1], r[:-1]
    f = np.divide((1 + beta**2) * p * r, beta**2 * p + r, out=np.zeros_like(p),
                  where=(beta**2 * p + r) > 0)
    return float(t[np.argmax(f)])


threshold = best_f_beta_threshold(yf_val.to_numpy(), calibrated.predict_proba(xf_val)[:, 1])
final = pd.DataFrame({
    "threshold 0.5": scores(yf_test.to_numpy(), proba_cal, 0.5),
    f"F2 threshold {threshold:.3f}": scores(yf_test.to_numpy(), proba_cal, threshold),
}).T
final

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    yf_test, (proba_cal >= threshold).astype(int),
    display_labels=["no diabetes", "diabetes"], cmap="Blues", values_format=",",
)
plt.title(f"Test confusion matrix at threshold {threshold:.3f}")
plt.show()

**Reading the result:** at threshold 0.5 a calibrated model flags almost nobody (diabetes is rare),
so recall is poor. The F2-chosen threshold flags a much larger group and catches most diabetic
people, at the cost of more false alarms — acceptable for a *screening* advisor that recommends seeing
a practitioner, not for diagnosis.

**Risk tiers used by the app** (relative to the base rate, stored with the model):
* `low` — probability below the training prevalence (~15 %)
* `moderate` — between 1× and 2.5× prevalence
* `high` — at least 2.5× prevalence (~38 %)

In [ ]:
cut_moderate, cut_high = prevalence, 2.5 * prevalence
tiers = pd.cut(proba_cal, [-0.01, cut_moderate, cut_high, 1.0], labels=["low", "moderate", "high"],
               right=False)
tier_table = pd.DataFrame({"tier": tiers, "diabetic": yf_test.to_numpy()}).groupby(
    "tier", observed=False)["diabetic"].agg(people="size", diabetic_share="mean")
tier_table["share_of_users"] = tier_table["people"] / tier_table["people"].sum()
tier_table

The tiers separate people well: in the test set roughly 5 % of `low`-tier people are diabetic,
about 25 % of `moderate` and about 50 % of `high` — so the tier is a meaningful input for tightening
a user's personal threshold.

## 8. Three-class model: none / prediabetes / diabetes

Multinomial logistic regression (softmax — the multi-class MLE) with class weights, on the 012 file.

In [ ]:
three = data["three_class"]
x3, y3 = three[FEATURES], three["Diabetes_012"].astype(int)
x3_tr, x3_te, y3_tr, y3_te = train_test_split(
    x3, y3, test_size=0.2, stratify=y3, random_state=RANDOM_STATE)

softmax = Pipeline([
    ("prep", linear_preprocess),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced")),
]).fit(x3_tr, y3_tr)
y3_pred = softmax.predict(x3_te)
print(classification_report(y3_te, y3_pred, target_names=["none", "prediabetes", "diabetes"],
                            digits=3))
ConfusionMatrixDisplay.from_predictions(
    y3_te, y3_pred, display_labels=["none", "prediabetes", "diabetes"], cmap="Blues",
    normalize="true", values_format=".2f")
plt.title("3-class confusion matrix (row-normalised)")
plt.show()

Prediabetes is almost indistinguishable from the other classes using survey answers alone (very low
precision): it is rare (< 2 %) and medically sits between the two. This is why the deployed model
uses the **binary** label (no diabetes vs. prediabetes-or-diabetes) and expresses severity through
the calibrated probability and tiers instead.

## 9. Which features drive the prediction?

Permutation importance: shuffle one feature at a time on held-out data and measure how much PR-AUC
drops. Model-agnostic and computed on the test set, so it reflects real predictive value.

In [ ]:
sample_idx = xf_test.sample(5000, random_state=RANDOM_STATE).index
importance = permutation_importance(
    calibrated, xf_test.loc[sample_idx], yf_test.loc[sample_idx],
    scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE,
)
imp = pd.Series(importance.importances_mean, index=FEATURES).sort_values()
plt.figure(figsize=(8, 6))
plt.barh(imp.index, imp.values, xerr=importance.importances_std[np.argsort(importance.importances_mean)],
         color="#4C72B0")
plt.title("Permutation importance (drop in PR-AUC)")
plt.tight_layout()
plt.show()
imp.sort_values(ascending=False).head(8)

## 10. Conclusions and how this plugs into the advisor

| Item | Result |
|---|---|
| Best model | Gradient boosting, isotonic-calibrated |
| Ranking quality | ROC-AUC ≈ 0.82 on the real-prevalence test set |
| Screening threshold | chosen by F2 on validation → high recall on diabetic users |
| Strongest signals | general health, high BP, BMI, age, high cholesterol |

**In the app** (`app/ml/`):
* `python -m app.ml.train` reproduces Section 7 and saves `models/risk_model.joblib` plus metadata
  (version, metrics, threshold, tier cut-offs).
* `RiskModel.load().assess(profile)` returns probability, tier and model version for each user at
  onboarding and whenever the profile changes (R1 batch update).
* The **threshold engine** (R2) starts from clinical glucose limits and tightens them for
  `moderate`/`high` tiers, within clamped safe bounds. Readings within 5 % of the limit trigger the
  voice advisory delivered as a Web Push notification (R3).

**Limitations**
* Self-reported US survey data from 2015 — may not transfer to Indian users without recalibration.
* No glucose, HbA1c or medication data; the model is a screening aid, **not a diagnosis**.
* Prediabetes cannot be reliably separated with these features.
* Fairness: performance should be checked per sex / age / income group before real use.